# Tutorial 5, Q1c: desert island knapsack

Release after the practical.

**The problem.** Table 1 of the sheet: eight items you could take to a desert island. You can carry 15 kg.
**Q1c:** as Q1a (each item at most once), but you must take at least 5 items.

| Item | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Weight (kg) | 2 | 3 | 2 | 6 | 6 | 8 | 3 | 5 |
| Survival value | 10 | 50 | 40 | 100 | 100 | 130 | 70 | 80 |

maximise $\sum_{i=1}^{8} s_i x_i$ subject to $\sum_{i=1}^{8} w_i x_i \le 15$, $\sum_{i=1}^{8} x_i \ge 5$, $x_i \in \{0, 1\}$. Because each $x_i$ is 0 or 1, their sum counts the items. In the code, items 1–8 are `x[0]` … `x[7]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Survival value **250**: items 1, 2, 3, 7 and 8 (`x[0]`, `x[1]`, `x[2]`, `x[6]`, `x[7]` = 1), exactly 15 kg. The answer sheet shows the same. A value printed as `-0.0` is zero (a rounding effect in the solver).

*Original script: `5 Source Files/Python/knapsack_tutorial_c.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A concrete model with one binary variable per item, as in Q1a.

In [ ]:
# Create a concrete model and define the binary variables: x[i] = 1 if item i+1 is packed
model = ConcreteModel()
model.x = Var(range(8), within=Binary)

The data from Table 1.

In [ ]:
# Define the objective function values, the weights and the capacity
val = [10, 50, 40, 100, 100, 130, 70, 80]  # survival value of items 1-8
weight = [2, 3, 2, 6, 6, 8, 3, 5]               # weight of items 1-8 (kg)
W = 15                                          # you can carry 15 kg

The objective: total survival value, maximised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(val[i] * model.x[i] for i in range(8)), sense=maximize)

The capacity: the packed items must not weigh more than 15 kg.

In [ ]:
# Define the capacity constraint: sum of w_i x_i <= W
model.capacity = Constraint(expr=sum(weight[i] * model.x[i] for i in range(8)) <= W)

The new rule: at least 5 items.

In [ ]:
# Define the constraint on the sum of binary variables: at least 5 items, sum of x_i >= 5
model.sum = Constraint(expr=sum(model.x[i] for i in range(8)) >= 5)

Create the HiGHS solver, print the model (`pprint()`: check it against the problem above), solve, and print the results (`display()`).

The HiGHS log appears between the two printouts (in the browser it is printed even without `tee=True`); its *Solving report* should say *Status Optimal*. In `display()`, look at the `Value` column of the variables and at the objective.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')
# Display the model
model.pprint()
# Solve the model
solver.solve(model)
# Display the results
model.display()

## What do you notice?

The value drops from 270 (Q1a) to 250. Could any other five items fit in 15 kg?

<details><summary>Answer</summary>

No. The five lightest items (1 and 3: 2 kg; 2 and 7: 3 kg; 8: 5 kg) weigh exactly 15 kg, and every other item
weighs at least 6 kg. So the rule "at least 5 items" leaves only one choice; the model finds it without being
told.
</details>